In [0]:
SHOW TABLES IN workspace.default;

database,tableName,isTemporary
default,bronze_category_translation,false
default,bronze_customers,false
default,bronze_geolocation,false
default,bronze_order_items,false
default,bronze_orders,false
default,bronze_payments,false
default,bronze_products,false
default,bronze_reviews,false
default,bronze_sellers,false
default,dim_customer,false


In [0]:
SELECT

    order_id,

    order_item_id,

    COUNT(*) AS row_count

FROM workspace.default.fact_sales

GROUP BY

    order_id,
    order_item_id

HAVING COUNT(*) > 1;

order_id,order_item_id,row_count


In [0]:
select * 
from workspace.default.fact_sales 
limit 10

order_id,order_item_id,customer_key,product_key,seller_key,date_key,payment_key,order_status,price,freight_value,item_sales,shipping_cost,total_item_value,order_total_value,total_payment,payment_difference,delivery_days,delivery_performance
04040ee654b248cdc512a68ecc83e4cc,1,30418,26336,2644,20180327,10,delivered,259.9,15.29,259.9,15.29,275.19,275.19,275.19,0.0,2,on_time
112f6cd3047e7f158607233e26206398,1,75801,17165,322,20180103,1,delivered,74.9,44.99,74.9,44.99,119.89,119.89,119.89,0.0,9,on_time
714e2644ab665027234ed7a6b3e82274,1,59258,20221,669,20170811,8,delivered,49.95,16.79,49.95,16.79,66.74,66.74,66.74,0.0,18,on_time
9d5dfd77cf32bd319a504f77a137491b,1,11713,29037,346,20180320,3,delivered,185.0,17.26,185.0,17.26,202.26,202.26,202.26,0.0,24,late
652c4ec4ac6a529e56176feba143e388,1,54273,25492,1875,20170424,4,delivered,195.0,15.54,195.0,15.54,210.54,210.54,210.54,0.0,14,on_time
762accd1f919295f51f3b39e314cee12,1,84218,28829,1639,20170529,9,delivered,139.0,15.73,139.0,15.73,154.73,154.73,154.73,0.0,8,on_time
0b7260709eb6a2884329b5ec9ef2b2a6,1,88689,12831,796,20180501,3,delivered,24.9,7.39,24.9,7.39,32.29,32.29,32.29,0.0,2,on_time
1d85312d372335db21f7f178c2325c6c,1,4520,11304,2729,20180723,8,delivered,133.0,25.8,133.0,25.8,158.8,158.8,158.8,0.0,3,on_time
1950d777989f6a877539f53795b4c3c3,1,39896,29384,498,20180219,4,canceled,29.99,14.1,29.99,14.1,44.09,44.09,44.09,0.0,30,late
7a70b827ebc6ab85bd4e28739619bb2d,2,42905,5826,1671,20180808,12,delivered,335.9,2.24,335.9,2.24,338.14,650.14,650.13,-0.01,10,on_time


In [0]:
CREATE OR REPLACE TABLE workspace.default.fact_sales
USING DELTA
AS

SELECT

    -- Degenerate order identifiers
    s.order_id,

    s.order_item_id,

    -- Dimension keys
    c.customer_key,

    p.product_key,

    se.seller_key,

    d.date_key,

    pay.payment_key,

    -- Order status
    s.order_status,

    -- Measures
    s.price,

    s.freight_value,

    s.item_sales,

    s.shipping_cost,

    s.total_item_value,

    -- Order-level financial metrics
    s.order_total_value,

    s.total_payment,

    s.payment_difference,

    -- Delivery metrics
    s.delivery_days,

    -- Business classification
    s.delivery_performance

FROM workspace.default.transformed_sales s

LEFT JOIN workspace.default.dim_customer c

    ON s.customer_unique_id =
       c.customer_unique_id

LEFT JOIN workspace.default.dim_product p

    ON s.product_id =
       p.product_id

LEFT JOIN workspace.default.dim_seller se

    ON s.seller_id =
       se.seller_id

LEFT JOIN workspace.default.dim_date d

    ON s.order_date =
       d.date

LEFT JOIN workspace.default.dim_payment pay

    ON s.primary_payment_type =
       pay.payment_type

    AND s.primary_payment_installments =
        pay.payment_installments;

num_affected_rows,num_inserted_rows


In [0]:
create or replace table
workspace.default.dim_payment
using delta as
select
row_number() over(
    order by payment_type,payment_installments
) as payment_key,
payment_type,
payment_installments
from (
    select distinct payment_type,payment_installments
    from workspace.default.silver_payments
    where payment_type is not null
);

num_affected_rows,num_inserted_rows


In [0]:
select *
from workspace.default.dim_date
order by date
limit 10;

date_key,date,year,quarter,month,month_name,day,day_of_week,day_name
20160904,2016-09-04,2016,3,9,September,4,1,Sunday
20160905,2016-09-05,2016,3,9,September,5,2,Monday
20160906,2016-09-06,2016,3,9,September,6,3,Tuesday
20160907,2016-09-07,2016,3,9,September,7,4,Wednesday
20160908,2016-09-08,2016,3,9,September,8,5,Thursday
20160909,2016-09-09,2016,3,9,September,9,6,Friday
20160910,2016-09-10,2016,3,9,September,10,7,Saturday
20160911,2016-09-11,2016,3,9,September,11,1,Sunday
20160912,2016-09-12,2016,3,9,September,12,2,Monday
20160913,2016-09-13,2016,3,9,September,13,3,Tuesday


In [0]:
create or replace table
workspace.default.dim_date
using delta as
select
cast(date_format(date,'yyyyMMdd') as int 
)as date_key,
date,
year(date) as year,
quarter(date) as quarter,
month(date) as month,
date_format(date,'MMMM') as month_name,
day(date) as day,
dayofweek(date) as day_of_week,
date_format(date,'EEEE') as day_name
from(
    select explode(
        sequence(
            ( 
            select min(order_date)
            from workspace.default.transformed_orders
            ),
            ( 
        select max(order_date)
            from workspace.default.transformed_orders
        ),
        interval 1 day
    )
) as date
);

num_affected_rows,num_inserted_rows


In [0]:
select min(order_date) as min_date,
max(order_date) as max_date
from workspace.default.transformed_orders

min_date,max_date
2016-09-04,2018-10-17


In [0]:
select *
from workspace.default.dim_seller
limit 10

seller_key,seller_id,seller_zip_code_prefix,seller_city,seller_state
1,0015a82c2db000af6aaaf3ae2ecb0532,9080,Santo Andre,SP
2,001cca7ae9ae17fb1caed9dfb1094831,29156,Cariacica,ES
3,001e6ad469a905060d959994f1b41e4f,24754,Sao Goncalo,RJ
4,002100f778ceb8431b7a1020ff7ab48f,14405,Franca,SP
5,003554e2dce176b5555353e4f3555ac8,74565,Goiania,GO
6,004c9cd9d87a3c30c522c48c4fc07416,14940,Ibitinga,SP
7,00720abe85ba0859807595bbf045a33b,7070,Guarulhos,SP
8,00ab3eff1b5192e5f1a63bcecfee11c8,4164,Sao Paulo,SP
9,00d8b143d12632bad99c0ad66ad52825,30170,Belo Horizonte,MG
10,00ee68308b45bc5e2660cd833c3f81cc,3333,Sao Paulo,SP


In [0]:
create or replace table 
workspace.default.dim_seller
using delta as
select
row_number() over(
    order by seller_id
) as seller_key,
seller_id,
max(seller_zip_code_prefix) as seller_zip_code_prefix,
max(seller_city) as seller_city,
max(seller_state) as seller_state
from workspace.default.transformed_sales
where seller_id is not null
group by seller_id

num_affected_rows,num_inserted_rows


In [0]:
create or replace table
workspace.default.dim_product
using delta as
select
row_number() over(
    order by product_id
) as product_key,
product_id,
max(product_category_name) as product_category_name,
max(product_category_name_english) as product_category_name_english,
max(product_name_lenght) as product_name_length,
max(product_description_lenght) as product_description_length,
max(product_photos_qty) as product_photos_qty,
max(product_weight_g) as product_weight_g,
max(product_length_cm) as product_length_cm,
max(product_height_cm) as product_height_cm,
max(product_width_cm) as product_width_cm
from workspace.default.transformed_sales
where product_id is not null
group by product_id;

num_affected_rows,num_inserted_rows


In [0]:
select *
from workspace.default.dim_customer
limit 10

customer_key,customer_unique_id,customer_city,customer_state,customer_zip_code_prefix
1,0000366f3b9a7992bf8c76cfdf3221e2,Cajamar,SP,7787
2,0000b849f77a49e4a4ce2b2a4ca5be3f,Osasco,SP,6053
3,0000f46a3911fa3c0805444483337064,Sao Jose,SC,88115
4,0000f6ccb0745a6a4b88665a16c9f078,Belem,PA,66812
5,0004aac84e0df4da2b147fca70cf8255,Sorocaba,SP,18040
6,0004bd2a26a76fe21f786e4fbd80607f,Sao Paulo,SP,5036
7,00050ab1314c0e55a6ca13cf7181fecf,Campinas,SP,13084
8,00053a61a98854899e70ed204dd4bafe,Curitiba,PR,80410
9,0005e1862207bf6ccc02e4228effd9a0,Teresopolis,RJ,25966
10,0005ef4cd20d2893f0d9fbd94d3c0d97,Sao Luis,MA,65060


In [0]:
create or replace table
workspace.default.dim_customer
using delta as
select
row_number() over(
    order by customer_unique_id
)
as customer_key,
customer_unique_id ,
max(customer_city)as customer_city,
max(customer_state)as customer_state,
max(customer_zip_code_prefix)as customer_zip_code_prefix
from workspace.default.transformed_sales
where customer_unique_id is not null
group by customer_unique_id

num_affected_rows,num_inserted_rows


In [0]:
DESCRIBE workspace.default.transformed_sales;

col_name,data_type,comment
order_id,string,null
order_item_id,int,null
order_status,string,null
customer_id,string,null
customer_unique_id,string,null
customer_city,string,null
customer_state,string,null
customer_zip_code_prefix,int,null
product_id,string,null
product_category_name,string,null


In [0]:
select * 
from workspace.default.transformed_sales
limit 10;

order_id,order_item_id,order_status,customer_id,customer_unique_id,customer_city,customer_state,customer_zip_code_prefix,product_id,product_category_name,product_category_name_english,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm,seller_id,seller_city,seller_state,seller_zip_code_prefix,order_date,order_purchase_timestamp,order_year,order_month,order_month_name,order_quarter,order_day,order_day_name,order_estimated_delivery_date,delivery_days,delivery_performance,price,freight_value,item_sales,shipping_cost,total_item_value,product_revenue,shipping_revenue,order_total_value,item_count,unique_products,unique_sellers,total_payment,payment_count,max_installments,primary_payment_type,primary_payment_installments,payment_difference
2807d0e504d6d4894d41672727bc139f,1,delivered,72ae281627a6102d9b3718528b420f8a,b8df986511d928829c3192c2ed081eba,Sao Paulo,SP,3323,6893767814d1ac82a81bcd365e1cc918,eletronicos,electronics,26,511,1,200,25,7,16,8b321bb669392f5163d04c59e235e066,Sao Paulo,SP,1212,2018-02-03,2018-02-03T20:37:35.000Z,2018,2,February,1,3,Saturday,2018-02-21T00:00:00.000Z,5,on_time,9.5,7.78,9.5,7.78,17.28,9.5,7.78,17.28,1,1,1,17.28,1,1,credit_card,1,0.0
ccbabeb0b02433bd0fcbac46e70339f2,1,delivered,c77ee2d8ba1614a4d489a44166894938,9c9cef121cb812cb301babddc2d8331e,Uberaba,MG,38067,89321f94e35fc6d7903d36f74e351d40,alimentos,food,59,982,1,150,17,13,13,16090f2ca825584b5a147ab24aa30c86,Atibaia,SP,12940,2018-02-19,2018-02-19T20:31:09.000Z,2018,2,February,1,19,Monday,2018-03-13T00:00:00.000Z,18,on_time,27.9,15.1,27.9,15.1,43.0,27.9,15.1,43.0,1,1,1,43.0,1,1,boleto,1,0.0
e4de6d53ecff736bc68804b0b6e9f635,1,delivered,9f6618c17568ac301465fe7ad056c674,e3bcfea9bab07b492391664fc1ffc28a,Antonio Cardoso,BA,44180,90b58782fdd04cb829667fcc41fb65f5,moveis_escritorio,office_furniture,34,794,1,7417,102,46,11,7c67e1448b00f6e969d365cea6b010ab,Itaquaquecetuba,SP,8577,2017-10-16,2017-10-16T14:56:50.000Z,2017,10,October,4,16,Monday,2017-11-21T00:00:00.000Z,23,on_time,179.99,51.13,179.99,51.13,231.12,179.99,51.13,231.12,1,1,1,231.12,1,1,boleto,1,0.0
cadbb3657dac2dbbd5b84b12e7b78aad,1,delivered,93ada7a24817edda9f4ab998fa823d16,cd148470c375939669971e8a032b16b4,Ribeirao Preto,SP,14091,9d2ff462feaaf88912539b8647e17ab4,informatica_acessorios,computers_accessories,42,315,1,813,32,16,16,00fc707aaaad2d31347cf883cd2dfe10,Maringa,PR,87025,2018-02-27,2018-02-27T12:55:42.000Z,2018,2,February,1,27,Tuesday,2018-03-29T00:00:00.000Z,17,on_time,394.9,14.89,394.9,14.89,409.79,394.9,14.89,409.79,1,1,1,409.79,1,1,boleto,1,0.0
d3d6788577c9592da441752e8a1dd5e3,1,delivered,8628fac2267e8c8804525da99c10ed0e,7973a6ba9c81ecaeb3d628c33c7c7c48,Palmas,PR,85555,7c1bd920dbdf22470b68bde975dd3ccf,beleza_saude,health_beauty,59,492,2,200,22,10,18,cc419e0650a3c5ba77189a1882b7556a,Santo Andre,SP,9015,2017-09-19,2017-09-19T22:17:15.000Z,2017,9,September,3,19,Tuesday,2017-10-13T00:00:00.000Z,21,on_time,58.99,17.66,58.99,17.66,76.65,58.99,17.66,76.65,1,1,1,76.65,1,7,credit_card,7,0.0
6a0a8bfbbe700284feb0845d95e0867f,1,delivered,68451b39b1314302c08c65a29f1140fc,781ae350edb16842380e81d7c7feb431,Rio De Janeiro,RJ,20740,f8a8f05a35976a91aed5cccc3992c357,moveis_decoracao,furniture_decor,63,418,1,1500,45,15,35,4a3ca9315b744ce9f8e9374361493884,Ibitinga,SP,14940,2017-11-22,2017-11-22T11:32:22.000Z,2017,11,November,4,22,Wednesday,2017-12-11T00:00:00.000Z,36,late,83.9,17.84,83.9,17.84,101.74,83.9,17.84,101.74,1,1,1,101.74,1,5,credit_card,5,0.0
0760a852e4e9d89eb77bf631eaaf1c84,1,invoiced,d2a79636084590b7465af8ab374a8cf5,c7f8d7b1fffc946d7069574f74c39f4e,Santo Amaro Da Imperatriz,SC,88140,1522589c64efd46731d3522568e5bc83,artigos_de_natal,christmas_supplies,35,415,4,550,37,10,37,28405831a29823802aa22c084cfd0649,Sao Paulo,SP,3644,2018-08-03,2018-08-03T17:44:42.000Z,2018,8,August,3,3,Friday,2018-08-21T00:00:00.000Z,null,not_delivered,35.0,15.35,35.0,15.35,50.35,35.0,15.35,50.35,1,1,1,50.35,1,1,boleto,1,0.0
e1da8361c76cab67aa3588